In [57]:
import pickle
# Path to the .pkl file
res = 'results_dbscan.pkl'

# Load the dictionary from the .pkl file
with open(res, 'rb') as file:
    results_dict = pickle.load(file)

In [58]:
import ast

def normalize_keys(results_dict):
    new_dict = {}
    for k, v in results_dict.items():
        if isinstance(k, str):
            if "T" in k:
                k = k.replace('T','')
            try:
                parsed_key = ast.literal_eval(k)  # converts '[30, 60, 600]' → [30, 60, 600]
                numeric_key = tuple(float(x) for x in parsed_key)
            except Exception:
                raise ValueError(f"Key {k} could not be parsed and converted.")
        else:
            numeric_key = tuple(float(x) for x in k)

        new_dict[numeric_key] = v
    return new_dict

# Apply the fix
results_dict = normalize_keys(results_dict)


In [59]:
results_dict

{(50.0,
  1.0,
  30.0,
  3.0):         Pred      Real  clusT_total  clusT_visits   Uq  Records  Stops  \
 0   0.632218  2.538483     0.108344      0.103833   21       72     68   
 1   0.423071  4.856498     0.196848      0.199764  106      178    152   
 2   0.575996  3.083867     0.079347      0.077711   32       80     77   
 3   0.443560  3.265216     0.030491      0.029923   18       36     28   
 4   0.612209  2.478416     0.172962      0.174606   16       60     60   
 6   0.610113  2.649790     0.033681      0.031933   21       55     55   
 7   0.480789  2.795111     0.021582      0.017896   12       21     20   
 8   0.502176  3.809634     0.111367      0.107484   51      117    115   
 9   0.735194  2.014798     0.181934      0.187890   23       66     65   
 11  0.603298  2.763494     0.150690      0.151285   24       71     71   
 12  0.630567  2.726229     0.094402      0.089840   29       77     74   
 13  0.713424  1.605044     0.084030      0.078489    7       62     6

In [60]:
list(range(-10,10))

[-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [61]:
import pandas as pd
import numpy as np
from itertools import combinations

def stop_count_by_user(df):
    return df['Stops']

def get_neighbors(param, all_params, step=1):
    # Ensure param is a list of numbers
    param = [float(p) for p in param]
    neighbors = []

    for i in range(len(param)):
        for delta in range(-step, step,10):
            new_param = param.copy()
            new_param[i] += delta
            new_param_tuple = tuple(new_param)

            if new_param_tuple in all_params:
                neighbors.append((i, new_param_tuple))

    return neighbors


def local_sensitivity_analysis(results_dict, step=1):
    all_params = set(results_dict.keys())
    sensitivity_records = []

    for param in results_dict:
        base_stops = stop_count_by_user(results_dict[param])

        neighbors = get_neighbors(list(param), all_params, step=step)

        for idx_changed, neighbor in neighbors:
            neighbor_stops = stop_count_by_user(results_dict[neighbor])
            # Ensure aligned index
            base, neighbor_val = base_stops.align(neighbor_stops, join='inner', fill_value=0)
            abs_diff = (base - neighbor_val).abs()
            rel_diff = abs_diff / (base + 1e-6)  # Avoid division by zero

            sensitivity_records.append({
                'base_param': param,
                'neighbor_param': neighbor,
                'changed_index': idx_changed,
                'mean_abs_change': abs_diff.mean(),
                'mean_rel_change': rel_diff.mean()
            })

    return pd.DataFrame(sensitivity_records)

# Run analysis
sensitivity_df = local_sensitivity_analysis(results_dict, step=100)  # you can change the step size here

In [62]:
sensitivity_df.groupby('base_param').mean().sort_values('mean_rel_change').head(30)

,changed_index,mean_abs_change,mean_rel_change
base_param,,,
"(500.0, 3.0, 500.0, 3.0)",1.500000,0.000000,0.000000
"(500.0, 1.0, 500.0, 10.0)",1.500000,0.000000,0.000000
"(500.0, 3.0, 300.0, 10.0)",1.500000,0.000000,0.000000
"(500.0, 1.0, 500.0, 3.0)",1.500000,0.000000,0.000000
"(500.0, 3.0, 300.0, 3.0)",1.500000,0.000000,0.000000
"(500.0, 1.0, 300.0, 10.0)",1.500000,0.000000,0.000000
"(500.0, 3.0, 500.0, 10.0)",1.500000,0.000000,0.000000
"(500.0, 1.0, 300.0, 3.0)",1.500000,0.000000,0.000000
"(500.0, 3.0, 500.0, 5.0)",1.800000,0.622222,0.018774


In [63]:
# For 2D or 3D parameter cases
import seaborn as sns

# Convert parameters to individual columns for plotting
param_df = stable_params.copy()
param_df[['p1', 'p2', 'p3']] = pd.DataFrame(param_df['params'].tolist(), index=param_df.index)

# Example: plot CV heatmap (if grid-like)
pivot = param_df.pivot_table(index='p1', columns='p2', values='cv')
sns.heatmap(pivot, annot=True, cmap='viridis')
plt.title("Coefficient of Variation of Stops")
plt.xlabel('Parameter 2')
plt.ylabel('Parameter 1')
plt.show()


ValueError: Columns must be same length as key

In [64]:
def local_sensitivity_analysis_with_preference(results_dict, step=10):
    from numpy.linalg import norm
    all_params = set(results_dict.keys())
    sensitivity_records = []

    for param in results_dict:
        base_stops = stop_count_by_user(results_dict[param])
        neighbors = get_neighbors(list(param), all_params, step=step)

        rel_changes = []

        for idx_changed, neighbor in neighbors:
            neighbor_stops = stop_count_by_user(results_dict[neighbor])
            base, neighbor_val = base_stops.align(neighbor_stops, join='inner', fill_value=0)
            rel_diff = (base - neighbor_val).abs() / (base + 1e-6)
            rel_changes.append(rel_diff.mean())

        if rel_changes:
            mean_rel_change = np.mean(rel_changes)
        else:
            mean_rel_change = np.nan  # or large number if you want to penalize this

        # Add preference for smaller parameter values (Euclidean norm or sum)
        simplicity_score = norm(param)  # or use sum(param) for linear penalty

        # Final score: weighted sum (you can adjust weights here)
        final_score = mean_rel_change + 0.001 * simplicity_score

        sensitivity_records.append({
            'param': param,
            'mean_rel_change': mean_rel_change,
            'param_norm': simplicity_score,
            'final_score': final_score
        })

    return pd.DataFrame(sensitivity_records).sort_values(by='final_score')

# Run and show
sensitivity_df = local_sensitivity_analysis_with_preference(results_dict, step=100)

In [65]:
sensitivity_df.sort_values('final_score')

,param,mean_rel_change,param_norm,final_score
24,"(50.0, 3.0, 50.0, 3.0)",0.037369,70.837843,0.108207
4,"(50.0, 1.0, 50.0, 3.0)",0.041559,70.781353,0.112340
26,"(50.0, 3.0, 50.0, 10.0)",0.041749,71.477269,0.113226
20,"(50.0, 3.0, 30.0, 3.0)",0.057061,58.463664,0.115525
6,"(50.0, 1.0, 50.0, 10.0)",0.044731,71.421285,0.116153
...,...,...,...,...
158,"(500.0, 3.0, 500.0, 10.0)",0.000000,707.183852,0.707184
157,"(500.0, 3.0, 500.0, 5.0)",0.018774,707.130822,0.725904
137,"(500.0, 1.0, 500.0, 5.0)",0.018795,707.125166,0.725920
139,"(500.0, 1.0, 500.0, 15.0)",0.022005,707.266569,0.729272
